In [ ]:
code = 'RED'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output\\'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def RED(bt, start_time, end_time, orderside, method, decay, sl, om):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        from_candle_close = True if method == 'CC' else False
        entry_time = start_dt

        ce_sl_price, ce_sl_flag, _, _, ce_sl_time, ce_pnl = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=sl, orderside=orderside, from_candle_close=from_candle_close)
        pe_sl_price, pe_sl_flag, _, _, pe_sl_time, pe_pnl = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=sl, orderside=orderside, from_candle_close=from_candle_close)
        
        ce_first_entrie = [ce_scrip, ce_price, ce_sl_price, ce_sl_flag, ce_sl_time, ce_pnl]
        pe_first_entrie = [pe_scrip, pe_price, pe_sl_price, pe_sl_flag, pe_sl_time, pe_pnl]

        ce_re_entries = []
        for re_no in range(max_re):
            if ce_sl_time and re_no < re_entries:
                start_dt = ce_sl_time
                ce_scrip, ce_price, _, start_dt = bt.get_strike(start_dt, end_dt, om=om, only='CE')
                
                if ce_scrip is None:
                    ce_sl_time = ''
                    ce_re_entries.extend(['', '', '', False, '', '', False, '', 0])
                    continue
                
                ce_decay_price, ce_decay_flag, ce_decay_time = bt.decay_check_single_leg(start_dt, end_dt, ce_scrip, decay=decay, from_candle_close=from_candle_close, orderside=orderside)

                if ce_decay_flag:
                    ce_sl_price, ce_sl_flag, _, _, ce_sl_time, ce_pnl = bt.sl_check_single_leg(ce_decay_time, end_dt, ce_scrip, o=(None if method == 'CC' else ce_decay_price), sl=sl, orderside=orderside, from_candle_close=from_candle_close)
                else:
                    ce_sl_price, ce_sl_flag, ce_sl_time, ce_pnl = '', False, '', 0

                ce_re_entries.extend([ce_scrip, ce_price, ce_decay_price, ce_decay_flag, ce_decay_time, ce_sl_price, ce_sl_flag, ce_sl_time, ce_pnl])
            else:
                ce_re_entries.extend(['', '', '', False, '', '', False, '', 0])
            
        pe_re_entries = []
        for re_no in range(max_re):
            if pe_sl_time and re_no < re_entries:
                start_dt = pe_sl_time
                pe_scrip, pe_price, _, start_dt = bt.get_strike(start_dt, end_dt, om=om, only='PE')
                
                if pe_scrip is None:
                    pe_sl_time = ''
                    pe_re_entries.extend(['', '', '', False, '', '', False, '', 0])
                    continue

                pe_decay_price, pe_decay_flag, pe_decay_time = bt.decay_check_single_leg(start_dt, end_dt, pe_scrip, decay=decay, from_candle_close=from_candle_close, orderside=orderside)

                if pe_decay_flag:
                    pe_sl_price, pe_sl_flag, _, _, pe_sl_time, pe_pnl = bt.sl_check_single_leg(pe_decay_time, end_dt, pe_scrip, o=(None if method == 'CC' else pe_decay_price), sl=sl, orderside=orderside, from_candle_close=from_candle_close)
                else:
                    pe_sl_price, pe_sl_flag, pe_sl_time, pe_pnl = '', False, '', 0

                pe_re_entries.extend([pe_scrip, pe_price, pe_decay_price, pe_decay_flag, pe_decay_time, pe_sl_price, pe_sl_flag, pe_sl_time, pe_pnl])
            else:
                pe_re_entries.extend(['', '', '', False, '', '', False, '', 0])
            
        return [code, bt.index, start_time, end_time, orderside, method, decay, sl, om, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price] + ce_first_entrie + ce_re_entries + pe_first_entrie + pe_re_entries
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, decay, sl, om])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            max_re, re_entries = 7, 7

            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_Method/P_Decay/P_SL/P_OM/Date/Day/DTE/EntryTime/Future')
            
            ce_log = '/CE.Strike/CE.Price/CE.SL.Price/CE0.SL.Flag/CE0.SL.Time/CE0.PNL'
            pe_log = '/PE.Strike/PE.Price/PE.SL.Price/PE0.SL.Flag/PE0.SL.Time/PE0.PNL'
            for re in range(1, max_re+1):
                ce_log += f'/CE{re}.Strike/CE{re}.Price/CE{re}.Decay.Price/CE{re}.Decay.Flag/CE{re}.Decay.Time/CE{re}.SL.Price/CE{re}.SL.Flag/CE{re}.SL.Time/CE{re}.PNL'
                pe_log += f'/PE{re}.Strike/PE{re}.Price/PE{re}.Decay.Price/PE{re}.Decay.Flag/PE{re}.Decay.Time/PE{re}.SL.Price/PE{re}.SL.Flag/PE{re}.SL.Time/PE{re}.PNL'
                
            log_cols = log_cols + ce_log + pe_log
            log_cols = log_cols.split('/')
            
            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)
                        
                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [RED(bt, row['entry_time'], row['exit_time'], row['orderside'], row['method'], row['decay'], row['sl'], row['om']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)

                    del bt
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))